# Reference Code Notebook
This notebook is intended to allow you to follow along with the code used in our video lessons. While this notebook generally mirrors the code in the videos, we have added certain lines that allow you to run the code cells independently. Examples of these added lines include the CREATE or REPLACE syntax and COPY INTO syntax that ensures tables and data exist where you would expect them to exist. 

Some code blocks will not execute without you making changes to them. These include code blocks that require Snowflake credentials to be input, a non-trial account, or code blocks that need to be run in an external Jupyter notebook. These codeblocks are individually noted in the readings before each set of code.

<h1>Module 1</h1>

### Overview: This module covers the foundational building blocks of Snowflake — worksheets, virtual warehouses, stages, databases, schemas, tables, views, and semi-structured data. You will learn how to navigate the Snowflake UI, write queries, manage compute resources, and organize your data.

## Section: Worksheets
### Lesson: Worksheets and Simple Example - Code

**Tips:**
- Worksheets are the primary interface for writing and running SQL in Snowflake.
- You can have multiple worksheets open at once — each maintains its own context (role, warehouse, database, schema).
- Use `--` for single-line comments in SQL.
- Results appear below the worksheet after execution. Only the last statement's result is displayed when running multiple statements together.

In [ ]:
%%sql -r Worksheets
---> set the Role
-- ACCOUNTADMIN has full privileges for creating objects
USE ROLE accountadmin;

---> set the Warehouse
-- COMPUTE_WH will execute our queries
USE WAREHOUSE compute_wh;

---> create the Tasty Bytes Database
CREATE OR REPLACE DATABASE tasty_bytes_sample_data;

---> create the Raw POS (Point-of-Sale) Schema
CREATE OR REPLACE SCHEMA tasty_bytes_sample_data.raw_pos;

---> create the Raw Menu Table
-- Defines the structure for menu data including a VARIANT column for JSON health metrics
CREATE OR REPLACE TABLE tasty_bytes_sample_data.raw_pos.menu
(
    menu_id NUMBER(19,0),
    menu_type_id NUMBER(38,0),
    menu_type VARCHAR(16777216),
    truck_brand_name VARCHAR(16777216),
    menu_item_id NUMBER(38,0),
    menu_item_name VARCHAR(16777216),
    item_category VARCHAR(16777216),
    item_subcategory VARCHAR(16777216),
    cost_of_goods_usd NUMBER(38,4),
    sale_price_usd NUMBER(38,4),
    menu_item_health_metrics_obj VARIANT
);

---> confirm the empty Menu table exists
SELECT * FROM tasty_bytes_sample_data.raw_pos.menu;

---> create the Stage referencing the Blob location and CSV File Format
-- External stage pointing to an S3 bucket; file_format tells Snowflake how to parse the files
CREATE OR REPLACE STAGE tasty_bytes_sample_data.public.blob_stage
url = 's3://sfquickstarts/tastybytes/'
file_format = (type = csv);

---> query the Stage to find the Menu CSV file
-- LIST shows files available at a given stage path
LIST @tasty_bytes_sample_data.public.blob_stage/raw_pos/menu/;

---> copy the Menu file into the Menu table
-- COPY INTO loads data from the stage into the target table
COPY INTO tasty_bytes_sample_data.raw_pos.menu
FROM @tasty_bytes_sample_data.public.blob_stage/raw_pos/menu/;

---> how many rows are in the table?
SELECT COUNT(*) AS row_count FROM tasty_bytes_sample_data.raw_pos.menu;

---> what do the top 10 rows look like?
SELECT TOP 10 * FROM tasty_bytes_sample_data.raw_pos.menu;

-- Count menu items per truck brand, ordered by most items first
SELECT TRUCK_BRAND_NAME, COUNT(*)
FROM tasty_bytes_sample_data.raw_pos.menu
GROUP BY 1
ORDER BY 2 DESC;

-- Count menu items grouped by both truck brand and menu type
SELECT
    TRUCK_BRAND_NAME,
    MENU_TYPE,
    COUNT(*)
FROM tasty_bytes_sample_data.raw_pos.menu
GROUP BY 1,2
ORDER BY 3 DESC;

## Section: Virtual Warehouses
### Lesson: Virtual Warehouses - Code

**Tips:**
- Virtual warehouses provide compute resources — they do NOT store data.
- Warehouses can be resized at any time (XS, S, M, L, XL, etc.) without downtime.
- `AUTO_SUSPEND` (in seconds) controls how long the warehouse stays active after the last query finishes. Setting it to 180 = 3 minutes.
- `AUTO_RESUME = TRUE` means the warehouse automatically starts when a query is submitted to it.
- Multi-cluster warehouses (`MAX_CLUSTER_COUNT > 1`) scale out to handle concurrency — useful when many users query simultaneously.
- `CREATE OR REPLACE` is idempotent — safe to run multiple times without error.
- Always drop warehouses you no longer need to avoid unnecessary credit consumption.

In [ ]:
%%sql -r dataframe_2
-- Filter menu items for a specific truck brand
SELECT 
   menu_item_name
FROM tasty_bytes_sample_data.raw_pos.menu
WHERE truck_brand_name = 'Freezing Point';

---> what is the profit on Mango Sticky Rice?
-- Calculate profit as the difference between sale price and cost of goods
SELECT 
   menu_item_name,
   (sale_price_usd - cost_of_goods_usd) AS profit_usd
FROM tasty_bytes_sample_data.raw_pos.menu
WHERE 1=1
AND truck_brand_name = 'Freezing Point'
AND menu_item_name = 'Mango Sticky Rice';

-- Create two new warehouses for different users/workloads
CREATE or REPLACE WAREHOUSE warehouse_dash;
CREATE or REPLACE WAREHOUSE warehouse_gilberto;

-- List all warehouses visible to the current role
SHOW WAREHOUSES;

-- Switch to a specific warehouse for query execution
USE WAREHOUSE warehouse_gilberto;

---> set warehouse size to medium
-- ALTER WAREHOUSE lets you resize without recreating; takes effect on next query
ALTER WAREHOUSE warehouse_dash SET warehouse_size=MEDIUM;

USE WAREHOUSE warehouse_dash;

-- Calculate profit for all menu items, sorted by highest profit first
SELECT
    menu_item_name,
   (sale_price_usd - cost_of_goods_usd) AS profit_usd
FROM tasty_bytes_sample_data.raw_pos.menu
ORDER BY 2 DESC;

---> set warehouse size to xsmall
-- Scale down when workload decreases to save credits
ALTER WAREHOUSE warehouse_dash SET warehouse_size=XSMALL;

---> drop warehouse
-- IF EXISTS prevents an error if the warehouse doesn't exist
DROP WAREHOUSE IF EXISTS warehouse_vino;

SHOW WAREHOUSES;

---> create a multi-cluster warehouse (max clusters = 3)
-- Multi-cluster warehouses auto-scale by adding clusters during concurrency spikes
CREATE or REPLACE WAREHOUSE warehouse_vino MAX_CLUSTER_COUNT = 3;

SHOW WAREHOUSES;

---> set the auto_suspend and auto_resume parameters
-- AUTO_SUSPEND: seconds of inactivity before suspending; AUTO_RESUME: auto-start on query
ALTER WAREHOUSE warehouse_dash SET AUTO_SUSPEND = 180 AUTO_RESUME = FALSE;

SHOW WAREHOUSES;


## Section: Stages and Basic Ingestion
### Lesson: Stages and Basic Ingestion - Code

**Tips:**
- A **stage** is a pointer to a location where data files are stored (internal to Snowflake or external like S3/GCS/Azure Blob).
- Use `LIST @stage_name` to see files available in a stage.
- Use `$1`, `$2`, etc. to reference columns positionally when querying files in a stage.
- `COPY INTO` is the primary command for bulk-loading data from a stage into a table.
- `FILE_FORMAT` defines how Snowflake should parse the file (CSV, JSON, Parquet, etc.).
- Always verify your data after loading with a `SELECT` statement.

In [ ]:
%%sql -r dataframe_24
USE ROLE accountadmin;

---> create tasty_bytes database
CREATE OR REPLACE DATABASE tasty_bytes;

---> create raw_pos schema
CREATE OR REPLACE SCHEMA tasty_bytes.raw_pos;

---> create raw_customer schema
CREATE OR REPLACE SCHEMA tasty_bytes.raw_customer;

---> create harmonized schema
CREATE OR REPLACE SCHEMA tasty_bytes.harmonized;

---> create analytics schema
CREATE OR REPLACE SCHEMA tasty_bytes.analytics;

---> create warehouses
CREATE OR REPLACE WAREHOUSE demo_build_wh
    WAREHOUSE_SIZE = 'xxxlarge'
    WAREHOUSE_TYPE = 'standard'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
COMMENT = 'demo build warehouse for tasty bytes assets';
    
CREATE OR REPLACE WAREHOUSE tasty_de_wh
    WAREHOUSE_SIZE = 'xsmall'
    WAREHOUSE_TYPE = 'standard'
    AUTO_SUSPEND = 60
    AUTO_RESUME = TRUE
    INITIALLY_SUSPENDED = TRUE
COMMENT = 'data engineering warehouse for tasty bytes';

USE WAREHOUSE tasty_de_wh;

---> file format creation
CREATE OR REPLACE FILE FORMAT tasty_bytes.public.csv_ff 
type = 'csv';

---> stage creation
CREATE OR REPLACE STAGE tasty_bytes.public.s3load
url = 's3://sfquickstarts/frostbyte_tastybytes/'
file_format = tasty_bytes.public.csv_ff;

---> list files in stage
ls @tasty_bytes.public.s3load;

---> country table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.country
(
    country_id NUMBER(18,0),
    country VARCHAR(16777216),
    iso_currency VARCHAR(3),
    iso_country VARCHAR(2),
    city_id NUMBER(19,0),
    city VARCHAR(16777216),
    city_population VARCHAR(16777216)
);

---> franchise table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.franchise 
(
    franchise_id NUMBER(38,0),
    first_name VARCHAR(16777216),
    last_name VARCHAR(16777216),
    city VARCHAR(16777216),
    country VARCHAR(16777216),
    e_mail VARCHAR(16777216),
    phone_number VARCHAR(16777216) 
);

---> location table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.location
(
    location_id NUMBER(19,0),
    placekey VARCHAR(16777216),
    location VARCHAR(16777216),
    city VARCHAR(16777216),
    region VARCHAR(16777216),
    iso_country_code VARCHAR(16777216),
    country VARCHAR(16777216)
);

---> menu table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.menu
(
    menu_id NUMBER(19,0),
    menu_type_id NUMBER(38,0),
    menu_type VARCHAR(16777216),
    truck_brand_name VARCHAR(16777216),
    menu_item_id NUMBER(38,0),
    menu_item_name VARCHAR(16777216),
    item_category VARCHAR(16777216),
    item_subcategory VARCHAR(16777216),
    cost_of_goods_usd NUMBER(38,4),
    sale_price_usd NUMBER(38,4),
    menu_item_health_metrics_obj VARIANT
);

---> truck table build 
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.truck
(
    truck_id NUMBER(38,0),
    menu_type_id NUMBER(38,0),
    primary_city VARCHAR(16777216),
    region VARCHAR(16777216),
    iso_region VARCHAR(16777216),
    country VARCHAR(16777216),
    iso_country_code VARCHAR(16777216),
    franchise_flag NUMBER(38,0),
    year NUMBER(38,0),
    make VARCHAR(16777216),
    model VARCHAR(16777216),
    ev_flag NUMBER(38,0),
    franchise_id NUMBER(38,0),
    truck_opening_date DATE
);

---> order_header table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_header
(
    order_id NUMBER(38,0),
    truck_id NUMBER(38,0),
    location_id FLOAT,
    customer_id NUMBER(38,0),
    discount_id VARCHAR(16777216),
    shift_id NUMBER(38,0),
    shift_start_time TIME(9),
    shift_end_time TIME(9),
    order_channel VARCHAR(16777216),
    order_ts TIMESTAMP_NTZ(9),
    served_ts VARCHAR(16777216),
    order_currency VARCHAR(3),
    order_amount NUMBER(38,4),
    order_tax_amount VARCHAR(16777216),
    order_discount_amount VARCHAR(16777216),
    order_total NUMBER(38,4)
);

---> order_detail table build
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_detail 
(
    order_detail_id NUMBER(38,0),
    order_id NUMBER(38,0),
    menu_item_id NUMBER(38,0),
    discount_id VARCHAR(16777216),
    line_number NUMBER(38,0),
    quantity NUMBER(5,0),
    unit_price NUMBER(38,4),
    price NUMBER(38,4),
    order_item_discount_amount VARCHAR(16777216)
);

---> customer loyalty table build
CREATE OR REPLACE TABLE tasty_bytes.raw_customer.customer_loyalty
(
    customer_id NUMBER(38,0),
    first_name VARCHAR(16777216),
    last_name VARCHAR(16777216),
    city VARCHAR(16777216),
    country VARCHAR(16777216),
    postal_code VARCHAR(16777216),
    preferred_language VARCHAR(16777216),
    gender VARCHAR(16777216),
    favourite_brand VARCHAR(16777216),
    marital_status VARCHAR(16777216),
    children_count VARCHAR(16777216),
    sign_up_date DATE,
    birthday_date DATE,
    e_mail VARCHAR(16777216),
    phone_number VARCHAR(16777216)
);

---> orders_v view
CREATE OR REPLACE VIEW tasty_bytes.harmonized.orders_v
    AS
SELECT 
    oh.order_id,
    oh.truck_id,
    oh.order_ts,
    od.order_detail_id,
    od.line_number,
    m.truck_brand_name,
    m.menu_type,
    t.primary_city,
    t.region,
    t.country,
    t.franchise_flag,
    t.franchise_id,
    f.first_name AS franchisee_first_name,
    f.last_name AS franchisee_last_name,
    l.location_id,
    cl.customer_id,
    cl.first_name,
    cl.last_name,
    cl.e_mail,
    cl.phone_number,
    cl.children_count,
    cl.gender,
    cl.marital_status,
    od.menu_item_id,
    m.menu_item_name,
    od.quantity,
    od.unit_price,
    od.price,
    oh.order_amount,
    oh.order_tax_amount,
    oh.order_discount_amount,
    oh.order_total
FROM tasty_bytes.raw_pos.order_detail od
JOIN tasty_bytes.raw_pos.order_header oh
    ON od.order_id = oh.order_id
JOIN tasty_bytes.raw_pos.truck t
    ON oh.truck_id = t.truck_id
JOIN tasty_bytes.raw_pos.menu m
    ON od.menu_item_id = m.menu_item_id
JOIN tasty_bytes.raw_pos.franchise f
    ON t.franchise_id = f.franchise_id
JOIN tasty_bytes.raw_pos.location l
    ON oh.location_id = l.location_id
LEFT JOIN tasty_bytes.raw_customer.customer_loyalty cl
    ON oh.customer_id = cl.customer_id;

---> loyalty_metrics_v view
CREATE OR REPLACE VIEW tasty_bytes.harmonized.customer_loyalty_metrics_v
    AS
SELECT 
    cl.customer_id,
    cl.city,
    cl.country,
    cl.first_name,
    cl.last_name,
    cl.phone_number,
    cl.e_mail,
    SUM(oh.order_total) AS total_sales,
    ARRAY_AGG(DISTINCT oh.location_id) AS visited_location_ids_array
FROM tasty_bytes.raw_customer.customer_loyalty cl
JOIN tasty_bytes.raw_pos.order_header oh
ON cl.customer_id = oh.customer_id
GROUP BY cl.customer_id, cl.city, cl.country, cl.first_name,
cl.last_name, cl.phone_number, cl.e_mail;

---> orders_v view
CREATE OR REPLACE VIEW tasty_bytes.analytics.orders_v
COMMENT = 'Tasty Bytes Order Detail View'
    AS
SELECT DATE(o.order_ts) AS date, * FROM tasty_bytes.harmonized.orders_v o;

---> customer_loyalty_metrics_v view
CREATE OR REPLACE VIEW tasty_bytes.analytics.customer_loyalty_metrics_v
COMMENT = 'Tasty Bytes Customer Loyalty Member Metrics View'
    AS
SELECT * FROM tasty_bytes.harmonized.customer_loyalty_metrics_v;

USE WAREHOUSE demo_build_wh;

---> country table load
COPY INTO tasty_bytes.raw_pos.country
FROM @tasty_bytes.public.s3load/raw_pos/country/;

---> franchise table load
COPY INTO tasty_bytes.raw_pos.franchise
FROM @tasty_bytes.public.s3load/raw_pos/franchise/;

---> location table load
COPY INTO tasty_bytes.raw_pos.location
FROM @tasty_bytes.public.s3load/raw_pos/location/;

---> menu table load
COPY INTO tasty_bytes.raw_pos.menu
FROM @tasty_bytes.public.s3load/raw_pos/menu/;

---> truck table load
COPY INTO tasty_bytes.raw_pos.truck
FROM @tasty_bytes.public.s3load/raw_pos/truck/;

---> customer_loyalty table load
COPY INTO tasty_bytes.raw_customer.customer_loyalty
FROM @tasty_bytes.public.s3load/raw_customer/customer_loyalty/;

---> order_header table load
COPY INTO tasty_bytes.raw_pos.order_header
FROM @tasty_bytes.public.s3load/raw_pos/order_header/;

---> order_detail table load
COPY INTO tasty_bytes.raw_pos.order_detail
FROM @tasty_bytes.public.s3load/raw_pos/order_detail/;

---> drop demo_build_wh
DROP WAREHOUSE IF EXISTS demo_build_wh;

USE WAREHOUSE tasty_de_wh;

SELECT file_name, error_count, status, last_load_time FROM snowflake.account_usage.copy_history
  ORDER BY last_load_time DESC
  LIMIT 10;

## Section: Tables
### Lesson: Tables - Code

**Tips:**
- `CREATE TABLE` defines a new table. Use `CREATE OR REPLACE TABLE` to avoid errors if it already exists.
- Snowflake supports standard data types: `VARCHAR`, `NUMBER`, `DATE`, `TIMESTAMP`, `BOOLEAN`, `VARIANT`, etc.
- `INSERT INTO` adds rows to a table. You can insert multiple rows in a single statement.
- `DESCRIBE TABLE` (or `DESC TABLE`) shows column names, types, and nullability.
- Tables are the primary storage objects in Snowflake — all data ultimately lives in tables.

In [ ]:
%%sql -r Tables
-- Ensure the test database and schema exist before using them
CREATE DATABASE IF NOT EXISTS TEST_DATABASE;
CREATE SCHEMA IF NOT EXISTS TEST_DATABASE.TEST_SCHEMA;

-- Set the active schema so we don't need fully-qualified names below
USE SCHEMA TEST_DATABASE.TEST_SCHEMA;

---> create a table – note that each column has a name and a data type
-- Demonstrates common Snowflake data types: NUMBER, VARCHAR, BOOLEAN, DATE, VARIANT, GEOGRAPHY
CREATE OR REPLACE TABLE TEST_TABLE (
	TEST_NUMBER NUMBER,
	TEST_VARCHAR VARCHAR,
	TEST_BOOLEAN BOOLEAN,
	TEST_DATE DATE,
	TEST_VARIANT VARIANT,
	TEST_GEOGRAPHY GEOGRAPHY
);

-- Query the newly created (empty) table
SELECT * FROM TEST_DATABASE.TEST_SCHEMA.TEST_TABLE;

---> insert a row into the table we just created
-- VALUES clause provides literal data for each column in order
INSERT INTO TEST_DATABASE.TEST_SCHEMA.TEST_TABLE
  VALUES
  (28, 'ha!', True, '2024-01-01', NULL, NULL);

-- Confirm the row was inserted
SELECT * FROM TEST_DATABASE.TEST_SCHEMA.TEST_TABLE;

---> drop the test table
DROP TABLE TEST_DATABASE.TEST_SCHEMA.TEST_TABLE;

---> see all tables in a particular schema
SHOW TABLES IN TEST_DATABASE.TEST_SCHEMA;

---> undrop the test table
-- Restores the dropped table via Time Travel
UNDROP TABLE TEST_DATABASE.TEST_SCHEMA.TEST_TABLE;

SHOW TABLES IN TEST_DATABASE.TEST_SCHEMA;

SHOW TABLES;

---> see table storage metadata from the Snowflake database
-- ACCOUNT_USAGE views provide historical metadata across the entire account
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.TABLE_STORAGE_METRICS;

SHOW TABLES;

---> here's an example of table we created previously
-- This table stores line-item details for each order
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.order_detail 
(
    order_detail_id NUMBER(38,0),
    order_id NUMBER(38,0),
    menu_item_id NUMBER(38,0),
    discount_id VARCHAR(16777216),
    line_number NUMBER(38,0),
    quantity NUMBER(5,0),
    unit_price NUMBER(38,4),
    price NUMBER(38,4),
    order_item_discount_amount VARCHAR(16777216)
);


In [ ]:
%%sql -r dataframe_18
-- Create a Dynamic Table that automatically refreshes every 1 minute
-- Dynamic Tables declaratively define transformations; Snowflake handles incremental refresh
CREATE OR REPLACE DYNAMIC TABLE tasty_bytes.analytics.tasty_bytes_revenue_by_city
  TARGET_LAG = '1 minute'
  WAREHOUSE = compute_wh
AS
SELECT
  l.city,
  SUM(o.order_total) AS total_revenue
FROM tasty_bytes.raw_pos.order_header o
JOIN tasty_bytes.raw_pos.location l
  ON o.location_id = l.location_id
GROUP BY l.city;

-- Query the dynamic table like any regular table
SELECT * FROM tasty_bytes.analytics.tasty_bytes_revenue_by_city;

## Section: Views
### Lesson: Views - Code

**Tips:**
- A **view** is a saved SQL query — it doesn't store data, it just references the underlying tables.
- Use `CREATE VIEW` to create a standard view. Use `CREATE SECURE VIEW` to hide the view definition from unauthorized users.
- Views simplify complex queries and provide a layer of abstraction over raw tables.
- A **materialized view** stores the query result physically and refreshes automatically — useful for expensive queries that don't change often.
- Views respect the permissions of the querying user, not the creator (unless they're secure views).

In [ ]:
%%sql -r Views
---> create the orders_v view – note the "CREATE VIEW view_name AS SELECT" syntax
-- A view is a saved query that acts like a virtual table; no data is stored separately
-- This view joins multiple raw tables to create a unified orders dataset
CREATE VIEW tasty_bytes.harmonized.orders_v
    AS
SELECT 
    oh.order_id,
    oh.truck_id,
    oh.order_ts,
    od.order_detail_id,
    od.line_number,
    m.truck_brand_name,
    m.menu_type,
    t.primary_city,
    t.region,
    t.country,
    t.franchise_flag,
    t.franchise_id,
    f.first_name AS franchisee_first_name,
    f.last_name AS franchisee_last_name,
    l.location_id,
    cl.customer_id,
    cl.first_name,
    cl.last_name,
    cl.e_mail,
    cl.phone_number,
    cl.children_count,
    cl.gender,
    cl.marital_status,
    od.menu_item_id,
    m.menu_item_name,
    od.quantity,
    od.unit_price,
    od.price,
    oh.order_amount,
    oh.order_tax_amount,
    oh.order_discount_amount,
    oh.order_total
FROM tasty_bytes.raw_pos.order_detail od
JOIN tasty_bytes.raw_pos.order_header oh
    ON od.order_id = oh.order_id
JOIN tasty_bytes.raw_pos.truck t
    ON oh.truck_id = t.truck_id
JOIN tasty_bytes.raw_pos.menu m
    ON od.menu_item_id = m.menu_item_id
JOIN tasty_bytes.raw_pos.franchise f
    ON t.franchise_id = f.franchise_id
JOIN tasty_bytes.raw_pos.location l
    ON oh.location_id = l.location_id
LEFT JOIN tasty_bytes.raw_customer.customer_loyalty cl
    ON oh.customer_id = cl.customer_id;

-- Verify the view has data
SELECT COUNT(*) FROM tasty_bytes.harmonized.orders_v;

-- Create a simple view that extracts just truck brand names from the menu
CREATE VIEW tasty_bytes.harmonized.brand_names 
    AS
SELECT truck_brand_name
FROM tasty_bytes.raw_pos.menu;

-- List all views in the current schema
SHOW VIEWS;

---> drop a view
DROP VIEW tasty_bytes.harmonized.brand_names;

SHOW VIEWS;

---> see metadata about a view
-- DESCRIBE VIEW shows columns, data types, and other properties
DESCRIBE VIEW tasty_bytes.harmonized.orders_v;


In [ ]:
%%sql -r dataframe_6
---> create a materialized view
-- Unlike regular views, materialized views store precomputed results for faster queries
-- Best for queries on large tables with expensive aggregations that don't change often
CREATE MATERIALIZED VIEW tasty_bytes.harmonized.brand_names_materialized 
    AS
SELECT DISTINCT truck_brand_name
FROM tasty_bytes.raw_pos.menu;

-- Query the materialized view (reads from precomputed storage, not the base table)
SELECT * FROM tasty_bytes.harmonized.brand_names_materialized;

-- SHOW VIEWS does not include materialized views
SHOW VIEWS;

-- Use SHOW MATERIALIZED VIEWS to see them
SHOW MATERIALIZED VIEWS;

---> see metadata about the materialized view we just made
DESCRIBE VIEW tasty_bytes.harmonized.brand_names_materialized;


In [ ]:
%%sql -r dataframe_8
-- DESCRIBE MATERIALIZED VIEW shows column definitions and metadata
DESCRIBE MATERIALIZED VIEW tasty_bytes.harmonized.brand_names_materialized;

---> drop the materialized view
DROP MATERIALIZED VIEW tasty_bytes.harmonized.brand_names_materialized;

---> Show dynamic tables and materialized views
SHOW DYNAMIC TABLES;
SHOW MATERIALIZED VIEWS;



In [ ]:
---> Reload menu data (the table was emptied by a previous CREATE OR REPLACE)
COPY INTO tasty_bytes.raw_pos.menu
FROM @tasty_bytes.public.s3load/raw_pos/menu/;

---> Create secure views
-- Secure views hide the view definition from consumers, useful for data sharing
-- The query text and logic are not visible even to users with access to the view
CREATE OR REPLACE SECURE VIEW tasty_bytes.analytics.brand_names_secure
AS
SELECT DISTINCT truck_brand_name
FROM tasty_bytes.raw_pos.menu;


In [ ]:

---> Query secure views
-- Querying a secure view works exactly like a regular view
SELECT * FROM tasty_bytes.analytics.brand_names_secure;

## Section: Semi-Structured Data
### Lesson: Semi-Structured Data - Code

**Tips:**
- Snowflake natively supports semi-structured data formats like JSON, Avro, Parquet, and XML.
- The `VARIANT` data type stores semi-structured data. Use `:` (colon) notation to traverse JSON keys (e.g., `column:key`).
- Use `FLATTEN` to explode arrays or nested objects into rows.
- Use `LATERAL FLATTEN` in a `FROM` clause to join flattened data back to the parent row.
- `PARSE_JSON()` converts a JSON string into a VARIANT. `TO_JSON()` does the reverse.
- Bracket notation (`['key']`) is an alternative to dot/colon notation for accessing nested fields.

In [ ]:
%%sql -r dataframe_7
---> see an example of a column with semi-structured (JSON) data
-- The VARIANT column menu_item_health_metrics_obj stores nested JSON objects
SELECT MENU_ITEM_NAME
    , MENU_ITEM_HEALTH_METRICS_OBJ
FROM tasty_bytes.RAW_POS.MENU;

-- DESCRIBE TABLE shows each column's name, data type, and nullability
DESCRIBE TABLE tasty_bytes.RAW_POS.MENU;


In [ ]:
%%sql -r dataframe_1

---> check out the data type for the menu_item_health_metrics_obj column – It's a VARIANT 
-- VARIANT is Snowflake's flexible type for storing semi-structured data (JSON, Avro, etc.)
CREATE or REPLACE TABLE tasty_bytes.raw_pos.menu
(
    menu_id NUMBER(19,0),
    menu_type_id NUMBER(38,0),
    menu_type VARCHAR(16777216),
    truck_brand_name VARCHAR(16777216),
    menu_item_id NUMBER(38,0),
    menu_item_name VARCHAR(16777216),
    item_category VARCHAR(16777216),
    item_subcategory VARCHAR(16777216),
    cost_of_goods_usd NUMBER(38,4),
    sale_price_usd NUMBER(38,4),
    menu_item_health_metrics_obj VARIANT
);

---> create the test_menu table with just a variant column in it, as a test
-- Casting a NUMBER to VARIANT demonstrates that any type can be stored as VARIANT
CREATE or REPLACE TABLE tasty_bytes.RAW_POS.TEST_MENU (cost_of_goods_variant)
AS SELECT cost_of_goods_usd::VARIANT
FROM tasty_bytes.RAW_POS.MENU;

---> notice that the column is of the VARIANT type
DESCRIBE TABLE tasty_bytes.RAW_POS.TEST_MENU;


In [ ]:
%%sql -r dataframe_5
---> Reload menu data (table was emptied by CREATE OR REPLACE in previous cell)
COPY INTO tasty_bytes.raw_pos.menu
FROM @tasty_bytes.public.s3load/raw_pos/menu/;

---> Recreate test_menu now that menu has data
CREATE OR REPLACE TABLE tasty_bytes.RAW_POS.TEST_MENU (cost_of_goods_variant)
AS SELECT cost_of_goods_usd::VARIANT
FROM tasty_bytes.RAW_POS.MENU;

---> but the typeof() function reveals the underlying data type
-- Even though the column is VARIANT, TYPEOF() shows the actual stored type (e.g., DECIMAL)
SELECT TYPEOF(cost_of_goods_variant) FROM tasty_bytes.raw_pos.test_menu;

---> Snowflake lets you perform operations based on the underlying data type
-- Arithmetic works on VARIANT columns if the underlying type is numeric
SELECT cost_of_goods_variant, cost_of_goods_variant*2.0 FROM tasty_bytes.raw_pos.test_menu;

-- Clean up the test table
DROP TABLE tasty_bytes.raw_pos.test_menu;

---> you can use the colon to pull out info from menu_item_health_metrics_obj
-- Colon notation (:) is used to access top-level keys in a VARIANT/JSON column
SELECT MENU_ITEM_HEALTH_METRICS_OBJ:menu_item_health_metrics FROM tasty_bytes.raw_pos.menu;

---> use typeof() to see the underlying type
-- For JSON objects, TYPEOF() returns 'OBJECT'
SELECT TYPEOF(MENU_ITEM_HEALTH_METRICS_OBJ) FROM tasty_bytes.raw_pos.menu;

-- Bracket notation ['key'] is an alternative to colon notation for accessing JSON keys
SELECT MENU_ITEM_HEALTH_METRICS_OBJ, MENU_ITEM_HEALTH_METRICS_OBJ['menu_item_id'] FROM tasty_bytes.raw_pos.menu;

# Module 2
## Section: Time Travel and Table Types
### Lesson: Time Travel - Code

**Overview:** Module 2 covers advanced data management features including Time Travel, table types, cloning, resource monitors, UDFs, stored procedures, and RBAC.

**Tips:**
- **Time Travel** lets you query historical data as it existed at a previous point in time.
- Use `AT(TIMESTAMP => ...)` to query data at a specific timestamp.
- Use `AT(OFFSET => -N)` to query data as it was N seconds ago.
- Use `BEFORE(STATEMENT => 'query_id')` to see data before a specific query ran.
- `DATA_RETENTION_TIME_IN_DAYS` controls how far back Time Travel can go (default: 1 day, max: 90 for Enterprise edition).
- Time Travel is invaluable for recovering from accidental UPDATEs or DELETEs.
- The `SET variable = value` syntax stores session variables; reference them with `$variable`.

In [ ]:
%%sql -r dataframe_9
SHOW TABLES;

---> create test_menu table for demonstration
-- CLONE creates an instant copy for testing without affecting the original
CREATE OR REPLACE TABLE TASTY_BYTES.RAW_POS.TEST_MENU CLONE TASTY_BYTES.RAW_POS.MENU;

---> set the data retention time to 90 days
-- Permanent tables support up to 90 days of Time Travel
ALTER TABLE TASTY_BYTES.RAW_POS.TEST_MENU SET DATA_RETENTION_TIME_IN_DAYS = 90;

SHOW TABLES;

---> set the data retention time to 1 day
-- Reduce retention to save storage costs when long Time Travel isn't needed
ALTER TABLE TASTY_BYTES.RAW_POS.TEST_MENU SET DATA_RETENTION_TIME_IN_DAYS = 1;

---> clone the truck table
-- Create a development copy to safely experiment without risking production data
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.truck_dev 
    CLONE tasty_bytes.raw_pos.truck;

-- View truck details from the dev clone
SELECT
    t.truck_id,
    t.year,
    t.make,
    t.model
FROM tasty_bytes.raw_pos.truck_dev t;
    
---> see how the age should have been calculated
-- Correct formula: current year minus the truck's model year
SELECT
    t.truck_id,
    t.year,
    t.make,
    t.model,
    (YEAR(CURRENT_DATE()) - t.year) AS truck_age
FROM tasty_bytes.raw_pos.truck_dev t;

---> record the most recent query_id, back when the data was still correct
-- LAST_QUERY_ID() captures the ID of the last executed query for Time Travel reference
SET good_data_query_id = LAST_QUERY_ID();

---> view the variable's value
SELECT $good_data_query_id;

---> record the time, back when the data was still correct
-- We'll use this timestamp later to recover data via Time Travel
SET good_data_timestamp = CURRENT_TIMESTAMP;

---> view the variable's value
SELECT $good_data_timestamp;

---> confirm that that worked
-- SHOW VARIABLES lists all session variables currently set
SHOW VARIABLES;

---> make the first mistake: calculating the truck's age incorrectly
-- BUG: Using division (/) instead of subtraction (-) gives wrong results
SELECT
    t.truck_id,
    t.year,
    t.make,
    t.model,
    (YEAR(CURRENT_DATE()) / t.year) AS truck_age
FROM tasty_bytes.raw_pos.truck_dev t;

---> make the second mistake: calculate age wrong, and overwrite the year!
-- DANGER: This UPDATE corrupts the year column with bad data
UPDATE tasty_bytes.raw_pos.truck_dev t
    SET t.year = (YEAR(CURRENT_DATE()) / t.year);

-- Confirm the data is now corrupted
SELECT
    t.truck_id,
    t.year,
    t.make,
    t.model
FROM tasty_bytes.raw_pos.truck_dev t;

---> select the data as of a particular timestamp
-- Time Travel: AT(TIMESTAMP =>) retrieves data as it existed at that specific moment
SELECT * FROM tasty_bytes.raw_pos.truck_dev
AT(TIMESTAMP => $good_data_timestamp);

SELECT $good_data_timestamp;

---> example code, without a timestamp inserted:
-- You can also hardcode a timestamp string (must cast to TIMESTAMP_LTZ)
-- SELECT * FROM tasty_bytes.raw_pos.truck_dev
-- AT(TIMESTAMP => '[insert timestamp]'::TIMESTAMP_LTZ);

--->example code, with a timestamp inserted (update timestamp to a recent value)
-- SELECT * FROM tasty_bytes.raw_pos.truck_dev
-- AT(TIMESTAMP => '2024-04-04 21:34:31.833 -0700'::TIMESTAMP_LTZ);

---> calculate the right offset
-- TIMESTAMPDIFF computes seconds between now and when the data was good
SELECT TIMESTAMPDIFF(second,CURRENT_TIMESTAMP,$good_data_timestamp);

---> Example code, without an offset inserted:
-- AT(OFFSET => -N) means "N seconds ago"
-- SELECT * FROM tasty_bytes.raw_pos.truck_dev
-- AT(OFFSET => -[WRITE OFFSET SECONDS PLUS A BIT]);

---> select the data as of a particular number of seconds back in time
---> (use offset based on the calculated difference above)
SELECT * FROM tasty_bytes.raw_pos.truck_dev
AT(OFFSET => -3);

SELECT $good_data_query_id;

---> select the data as of its state before a previous query was run
-- BEFORE(STATEMENT =>) retrieves data as it was just before a specific query executed
SELECT * FROM tasty_bytes.raw_pos.truck_dev
BEFORE(STATEMENT => $good_data_query_id);

## Section: Time Travel and Table Types
### Lesson: Permanent, Transient, and Temporary Tables - Code

**Tips:**
- **Permanent tables** (default) support Time Travel up to 90 days and have Fail-safe (7 days of additional recovery by Snowflake support).
- **Transient tables** (`CREATE TRANSIENT TABLE`) have Time Travel limited to 0-1 days and NO Fail-safe — good for staging/ETL data you can regenerate.
- **Temporary tables** (`CREATE TEMPORARY TABLE`) exist only for the session duration, have 0-1 day Time Travel, and no Fail-safe.
- Transient and temporary tables cost less storage because they skip Fail-safe.
- You cannot set `DATA_RETENTION_TIME_IN_DAYS` higher than 1 for transient or temporary tables.

In [ ]:
%%sql -r dataframe_10
-- Set context to the correct database and schema
USE DATABASE TASTY_BYTES;
USE SCHEMA RAW_POS;

---> drop truck_dev if not dropped previously
DROP TABLE IF EXISTS TASTY_BYTES.RAW_POS.TRUCK_DEV;

---> create a transient table
-- Transient tables have no Fail-safe period, reducing storage costs
-- Max Time Travel retention is 1 day (vs. 90 days for permanent tables)
CREATE OR REPLACE TRANSIENT TABLE TASTY_BYTES.RAW_POS.TRUCK_TRANSIENT
    CLONE TASTY_BYTES.RAW_POS.TRUCK;

---> create a temporary table
-- Temporary tables only exist for the duration of the session and are not visible to other sessions
CREATE OR REPLACE TEMPORARY TABLE TASTY_BYTES.RAW_POS.TRUCK_TEMPORARY
    CLONE TASTY_BYTES.RAW_POS.TRUCK;

---> show tables that start with the word TRUCK
-- LIKE pattern filters results; notice the 'kind' column shows TRANSIENT vs TEMPORARY vs standard
SHOW TABLES LIKE 'TRUCK%' IN TASTY_BYTES.RAW_POS;


In [ ]:
%%sql -r dataframe_22
---> attempt (successfully) to set the data retention time to 90 days for the standard table
-- DATA_RETENTION_TIME_IN_DAYS controls how far back Time Travel can go
-- Standard (permanent) tables support up to 90 days
ALTER TABLE TASTY_BYTES.RAW_POS.TRUCK SET DATA_RETENTION_TIME_IN_DAYS = 90;

---> transient tables support a max retention of 1 day
ALTER TABLE TASTY_BYTES.RAW_POS.TRUCK_TRANSIENT SET DATA_RETENTION_TIME_IN_DAYS = 1;

---> temporary tables support a max retention of 1 day
ALTER TABLE TASTY_BYTES.RAW_POS.TRUCK_TEMPORARY SET DATA_RETENTION_TIME_IN_DAYS = 1;

-- Verify the retention settings in the output
SHOW TABLES LIKE 'TRUCK%' IN TASTY_BYTES.RAW_POS;


In [ ]:
%%sql -r dataframe_23
---> set the data retention time to 0 days for the transient table
-- Setting to 0 disables Time Travel entirely (no historical data recovery)
ALTER TABLE TASTY_BYTES.RAW_POS.TRUCK_TRANSIENT SET DATA_RETENTION_TIME_IN_DAYS = 0;

---> set the data retention time to 0 days for the temporary table
ALTER TABLE TASTY_BYTES.RAW_POS.TRUCK_TEMPORARY SET DATA_RETENTION_TIME_IN_DAYS = 0;

-- Confirm retention is now 0 for both tables
SHOW TABLES LIKE 'TRUCK%' IN TASTY_BYTES.RAW_POS;

## Section: Cloning and Resource Monitors
### Lesson: Cloning - Code

**Tips:**
- `CLONE` creates a zero-copy clone — it's instant and doesn't duplicate storage until data diverges.
- You can clone databases, schemas, and tables.
- Clones are independent: changes to the clone don't affect the original, and vice versa.
- Cloning is great for creating dev/test environments from production data without extra cost.
- Combine with Time Travel: `CLONE ... AT(TIMESTAMP => ...)` to clone historical data.

In [ ]:
%%sql -r dataframe_11
---> create a clone of the truck table
-- CLONE creates a zero-copy clone: metadata points to the same micro-partitions (no extra storage initially)
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.truck_clone 
    CLONE tasty_bytes.raw_pos.truck;

/* look at metadata for the truck and truck_clone tables from the table_storage_metrics view in the information_schema */
-- Both tables initially share storage due to zero-copy cloning
SELECT * FROM TASTY_BYTES.INFORMATION_SCHEMA.TABLE_STORAGE_METRICS
WHERE TABLE_NAME = 'TRUCK_CLONE' OR TABLE_NAME = 'TRUCK';

/* look at metadata for the truck and truck_clone tables from the tables view in the information_schema */
SELECT * FROM TASTY_BYTES.INFORMATION_SCHEMA.TABLES
WHERE TABLE_NAME = 'TRUCK_CLONE' OR TABLE_NAME = 'TRUCK';

---> insert the truck table into the clone (thus doubling the clone's size!)
-- After modification, the clone diverges and uses its own storage for new data
INSERT INTO tasty_bytes.raw_pos.truck_clone
SELECT * FROM tasty_bytes.raw_pos.truck;

---> now use the tables view to look at metadata for the truck and truck_clone tables again
-- Notice the clone now has double the rows
SELECT * FROM TASTY_BYTES.INFORMATION_SCHEMA.TABLES
WHERE TABLE_NAME = 'TRUCK_CLONE' OR TABLE_NAME = 'TRUCK';

---> clone a schema
-- All objects within the schema are cloned (tables, views, etc.)
CREATE OR REPLACE SCHEMA tasty_bytes.raw_pos_clone
CLONE tasty_bytes.raw_pos;

---> clone a database
-- Clones all schemas and objects within the database
CREATE OR REPLACE DATABASE tasty_bytes_clone
CLONE tasty_bytes;

---> clone a table based on an offset (so the table as it was at a certain interval in the past)
-- AT(OFFSET => -60*10) means "as it was 600 seconds (10 minutes) ago" using Time Travel
CREATE OR REPLACE TABLE tasty_bytes.raw_pos.truck_clone_time_travel 
    CLONE tasty_bytes.raw_pos.truck AT(OFFSET => -60*10);

SELECT * FROM tasty_bytes.raw_pos.truck_clone_time_travel;

## Section: Cloning and Resource Monitors
### Lesson: Resource Monitors - Code

**Tips:**
- **Resource monitors** track and control credit usage for warehouses.
- Set `CREDIT_QUOTA` to define the monthly credit limit.
- Use triggers at different thresholds: `NOTIFY` (alert only), `SUSPEND` (stop new queries), `SUSPEND_IMMEDIATELY` (kill running queries).
- Resource monitors can be assigned to one or more warehouses, or to the entire account.
- Only `ACCOUNTADMIN` can create resource monitors by default.
- They reset at the start of each billing cycle (or custom frequency).

In [ ]:
%%sql -r dataframe_12
---> create a resource monitor
-- Resource monitors track credit consumption and can notify or suspend warehouses at thresholds
CREATE RESOURCE MONITOR tasty_test_rm
WITH 
    CREDIT_QUOTA = 20 -- 20 credits allowed per period
    FREQUENCY = daily -- reset the monitor daily
    START_TIMESTAMP = immediately -- begin tracking immediately
    TRIGGERS 
        ON 80 PERCENT DO NOTIFY -- notify accountadmins at 80%
        ON 100 PERCENT DO SUSPEND -- suspend warehouse at 100%, let running queries finish
        ON 110 PERCENT DO SUSPEND_IMMEDIATE; -- suspend warehouse and cancel all queries at 110%

---> see all resource monitors
SHOW RESOURCE MONITORS;

---> assign a resource monitor to a warehouse
-- This links the monitor to the warehouse so it tracks that warehouse's credit usage
ALTER WAREHOUSE tasty_de_wh SET RESOURCE_MONITOR = tasty_test_rm;

SHOW RESOURCE MONITORS;

---> change the credit quota on a resource monitor
-- You can adjust limits without recreating the monitor
ALTER RESOURCE MONITOR tasty_test_rm
  SET CREDIT_QUOTA=30;

SHOW RESOURCE MONITORS;


In [ ]:
%%sql -r dataframe_21
---> drop a resource monitor
-- Removes the monitor; the warehouse will no longer be tracked by this monitor
DROP RESOURCE MONITOR tasty_test_rm;

SHOW RESOURCE MONITORS;

## Section: User Defined Functions (UDFs)
### Lesson: UDFs and UDTFs - Code

**Tips:**
- **UDFs** (User Defined Functions) return a single scalar value per row. **UDTFs** (User Defined Table Functions) return a table.
- UDFs can be written in SQL, JavaScript, Python, or Java.
- Call a scalar UDF in a `SELECT`: `SELECT my_function(column)`.
- Call a UDTF with `TABLE()`: `SELECT * FROM TABLE(my_udtf(args))`.
- Use `CREATE OR REPLACE FUNCTION` to avoid "already exists" errors on re-runs.
- Python UDFs require specifying `LANGUAGE PYTHON`, `RUNTIME_VERSION`, and a `HANDLER` function name.
- UDFs are great for encapsulating reusable business logic that can be shared across queries.

In [ ]:
%%sql -r dataframe_13
---> here's an example of a function in action!
-- ABS() returns the absolute value of a number
SELECT ABS(-14);

---> here's another example of a function in action!
-- UPPER() converts a string to uppercase
SELECT UPPER('upper');

---> see all functions
-- Lists both built-in and user-defined functions
SHOW FUNCTIONS;

-- Find the highest sale price on the menu
SELECT MAX(SALE_PRICE_USD) FROM TASTY_BYTES.RAW_POS.MENU;

---> use a particular database
USE DATABASE TASTY_BYTES;

---> create the max_menu_price function
-- A scalar UDF (User-Defined Function) that returns a single value
-- No arguments needed – it always queries the same table
CREATE OR REPLACE FUNCTION max_menu_price()
  RETURNS NUMBER(5,2)
  AS
  $$
    SELECT MAX(SALE_PRICE_USD) FROM TASTY_BYTES.RAW_POS.MENU
  $$
  ;

---> run the max_menu_price function by calling it in a select statement
SELECT max_menu_price();

SHOW FUNCTIONS;

---> create a new function, but one that takes in an argument
-- This UDF accepts a conversion rate and returns the max price in the new currency
CREATE OR REPLACE FUNCTION max_menu_price_converted(USD_to_new NUMBER)
  RETURNS NUMBER(5,2)
  AS
  $$
    SELECT USD_TO_NEW*MAX(SALE_PRICE_USD) FROM TASTY_BYTES.RAW_POS.MENU
  $$
  ;

-- Convert max price to another currency (e.g., 1.35 for USD to EUR)
SELECT max_menu_price_converted(1.35);

---> create a Python function
-- UDFs can be written in Python; the handler specifies which Python function to call
-- This function "winsorizes" a value: clips it to upper/lower bounds
CREATE OR REPLACE FUNCTION winsorize (val NUMERIC, up_bound NUMERIC, low_bound NUMERIC)
returns NUMERIC
language python
runtime_version = '3.11'
handler = 'winsorize_py'
AS
$$
def winsorize_py(val, up_bound, low_bound):
    if val > up_bound:
        return up_bound
    elif val < low_bound:
        return low_bound
    else:
        return val
$$;

---> run the Python function
-- Input 12.0 exceeds upper bound 11.0, so result is clipped to 11.0
SELECT winsorize(12.0, 11.0, 4.0);

---> here's the reference UDF we're going to work off of as we make our UDTF
CREATE OR REPLACE FUNCTION max_menu_price()
  RETURNS NUMBER(5,2)
  AS
  $$
    SELECT MAX(SALE_PRICE_USD) FROM TASTY_BYTES.RAW_POS.MENU
  $$
  ;

USE DATABASE TASTY_BYTES;
  
---> create a user-defined table function
-- UDTFs return a TABLE (multiple rows/columns) instead of a single scalar value
-- This UDTF returns all menu items above a given price threshold
CREATE OR REPLACE FUNCTION menu_prices_above(price_floor NUMBER)
  RETURNS TABLE (item VARCHAR, price NUMBER)
  AS
  $$
    SELECT MENU_ITEM_NAME, SALE_PRICE_USD 
    FROM TASTY_BYTES.RAW_POS.MENU
    WHERE SALE_PRICE_USD > price_floor
    ORDER BY 2 DESC
  $$
  ;
  
---> now you can see it in the list of all functions!
SHOW FUNCTIONS;

---> run the UDTF to see what the output looks like
-- TABLE() wraps the UDTF call so it can be used in the FROM clause
SELECT * FROM TABLE(menu_prices_above(15));

---> you can use a where clause on the result
-- ILIKE is a case-insensitive LIKE; % is the wildcard
SELECT * FROM TABLE(menu_prices_above(15)) 
WHERE ITEM ILIKE '%CHICKEN%';

## Section: Stored Procedures
### Lesson: Stored Procedures - Code

**Tips:**
- **Stored procedures** can perform DDL/DML operations (CREATE, INSERT, UPDATE, DELETE) — unlike UDFs which are read-only.
- Procedures are called with `CALL procedure_name(args)`.
- They can be written in SQL (Snowflake Scripting), JavaScript, Python, or Java.
- Use procedures for administrative tasks, data transformations, or complex multi-step workflows.
- Procedures can run with the caller's rights (`EXECUTE AS CALLER`) or the owner's rights (`EXECUTE AS OWNER`).
- Unlike UDFs, stored procedures cannot be used inside a `SELECT` statement.

In [ ]:
%%sql -r dataframe_14
---> list all procedures
SHOW PROCEDURES;

-- Preview order data to understand the table structure
SELECT * FROM TASTY_BYTES_CLONE.RAW_POS.ORDER_HEADER
LIMIT 100;

---> see the latest and earliest order timestamps so we can determine what we want to delete
SELECT MAX(ORDER_TS), MIN(ORDER_TS) FROM TASTY_BYTES_CLONE.RAW_POS.ORDER_HEADER;

---> save the max timestamp
-- Session variables (SET) store values for reuse later in the session
SET max_ts = (SELECT MAX(ORDER_TS) FROM TASTY_BYTES_CLONE.RAW_POS.ORDER_HEADER);

-- Reference a session variable with the $ prefix
SELECT $max_ts;

-- DATEADD subtracts 180 days from the max timestamp to find the cutoff
SELECT DATEADD('DAY',-180,$max_ts);

---> determine the necessary cutoff to go back 180 days
SET cutoff_ts = (SELECT DATEADD('DAY',-180,$max_ts));

---> note how you can use the cutoff_ts variable in the WHERE clause
-- This previews what will be deleted (everything older than 180 days from the latest order)
SELECT MAX(ORDER_TS) FROM TASTY_BYTES_CLONE.RAW_POS.ORDER_HEADER
WHERE ORDER_TS < $cutoff_ts;

USE DATABASE TASTY_BYTES;

---> create your procedure
-- Stored procedures encapsulate reusable logic; this one deletes orders older than 180 days
-- RETURNS BOOLEAN indicates it returns a success/failure flag
-- LANGUAGE SQL means the body is written in Snowflake Scripting (SQL-based procedural language)
CREATE OR REPLACE PROCEDURE delete_old()
RETURNS BOOLEAN
LANGUAGE SQL
AS
$$
DECLARE
  max_ts TIMESTAMP;
  cutoff_ts TIMESTAMP;
BEGIN
  -- Find the most recent order timestamp
  max_ts := (SELECT MAX(ORDER_TS) FROM TASTY_BYTES_CLONE.RAW_POS.ORDER_HEADER);
  -- Calculate the cutoff: 180 days before the most recent order
  cutoff_ts := (SELECT DATEADD('DAY',-180,:max_ts));
  -- Delete all orders older than the cutoff
  DELETE FROM TASTY_BYTES_CLONE.RAW_POS.ORDER_HEADER
  WHERE ORDER_TS < :cutoff_ts;
END;
$$
;

SHOW PROCEDURES;

---> see information about your procedure
-- DESCRIBE PROCEDURE shows parameters, return type, and body
DESCRIBE PROCEDURE delete_old();

---> run your procedure
-- CALL executes the stored procedure
CALL DELETE_OLD();

---> confirm that that made a difference
SELECT MIN(ORDER_TS) FROM TASTY_BYTES_CLONE.RAW_POS.ORDER_HEADER;

---> it did! We deleted everything from before the cutoff timestamp
SELECT $cutoff_ts;

## Section: Role-based Access Control (RBAC)
### Lesson: Role-based Access Control - Code

**Tips:**
- **RBAC** (Role-Based Access Control) is Snowflake's security model — all permissions are granted to roles, and roles are granted to users.
- Use `CREATE ROLE` to define a new role. Use `GRANT` to assign privileges to that role.
- The built-in role hierarchy: `ACCOUNTADMIN` > `SECURITYADMIN` > `USERADMIN` > `SYSADMIN` > `PUBLIC`.
- `SHOW GRANTS TO ROLE role_name` reveals what privileges a role currently holds.
- A user must be granted a role before they can `USE ROLE role_name`.
- Follow the principle of least privilege — grant only the permissions needed for each role's function.
- Use `GRANT ROLE child_role TO ROLE parent_role` to build custom role hierarchies.

In [ ]:
%%sql -r dataframe_15
-- Must be ACCOUNTADMIN to create roles and manage privileges
USE ROLE accountadmin;

---> create a role
-- Roles are the primary way to control access in Snowflake (RBAC model)
CREATE ROLE IF NOT EXISTS tasty_de;

---> see what privileges this new role has
-- A newly created role has no privileges until explicitly granted
SHOW GRANTS TO ROLE tasty_de;

---> see what privileges an auto-generated role has
-- ACCOUNTADMIN is the top-level role with all privileges
SHOW GRANTS TO ROLE accountadmin;

---> grant a role to a specific user. The username will need to be filled in with a real username
-- This allows the user to USE ROLE tasty_de
GRANT ROLE tasty_de TO USER [username];

---> use a role
USE ROLE tasty_de;

---> try creating a warehouse with this new role
-- This will FAIL because tasty_de doesn't have CREATE WAREHOUSE privilege yet
CREATE WAREHOUSE tasty_de_test;

-- Switch back to accountadmin to grant the needed privilege
USE ROLE accountadmin;

---> grant the create warehouse privilege to the tasty_de role
-- GRANT ON ACCOUNT gives account-level privileges to the role
GRANT CREATE WAREHOUSE ON ACCOUNT TO ROLE tasty_de;

---> show all of the privileges the tasty_de role has
SHOW GRANTS TO ROLE tasty_de;

-- Switch back to tasty_de to test the new privilege
USE ROLE tasty_de;

---> test to see whether tasty_de can create a warehouse
-- This should now succeed after the GRANT
CREATE WAREHOUSE tasty_de_test;

---> learn more about the privileges each of the following auto-generated roles has
-- SECURITYADMIN: manages grants and roles
SHOW GRANTS TO ROLE securityadmin;

-- USERADMIN: manages users and roles
SHOW GRANTS TO ROLE useradmin;

-- SYSADMIN: manages warehouses, databases, and schemas
SHOW GRANTS TO ROLE sysadmin;

-- PUBLIC: default role granted to all users; minimal privileges
SHOW GRANTS TO ROLE public;

## Section: Snowpark DataFrames and VS Code Extension
### Lesson: Snowpark DataFrames - Code

**Tips:**
- **Snowpark** lets you write Python (or Scala/Java) that executes inside Snowflake — data never leaves the platform.
- A `DataFrame` is a lazy reference to a table or query; operations are not executed until you call `.show()`, `.collect()`, or `.count()`.
- Use `session.table("db.schema.table")` to create a DataFrame from an existing table.
- Chain `.filter()`, `.select()`, `.group_by()`, and `.agg()` for transformations — similar to PySpark or pandas.
- `session.sql("SELECT ...")` returns a DataFrame from raw SQL if you prefer writing SQL directly.
- In Snowflake Notebooks, use `get_active_session()` instead of building a session manually with credentials.

In [ ]:
# Get the active Snowpark session (works inside Snowflake Notebooks)
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark.functions import col, sum as sum_, avg, count
session = get_active_session()

# Create a DataFrame reference to the menu table
df = session.table("TASTY_BYTES_SAMPLE_DATA.raw_pos.menu")

# Display the first 10 rows (default)
df.show()

# .collect() pulls ALL rows into local memory as a list of Row objects
rows = df.collect()
print(f"Row count: {len(rows)}")
print(rows[0])

from snowflake.snowpark.functions import col

# Filter the DataFrame to only include rows where truck brand is "Freezing Point"
df_freezing = df.filter(col("TRUCK_BRAND_NAME") == "Freezing Point")
df_freezing.show()

# Select only specific columns from the filtered DataFrame
df_selected = df_freezing.select(
    col("MENU_ITEM_NAME"),
    col("SALE_PRICE_USD")
)
df_selected.show()

# Method chaining: filter and select in one fluent expression
df_result = (
    df
    .filter(col("TRUCK_BRAND_NAME") == "Freezing Point")
    .select(col("MENU_ITEM_NAME"), col("SALE_PRICE_USD"))
)
df_result.show()

# Alternative: use raw SQL via session.sql() for the same result
df_sql = session.sql("""
    SELECT MENU_ITEM_NAME, SALE_PRICE_USD
    FROM TASTY_BYTES_SAMPLE_DATA.raw_pos.menu
    WHERE TRUCK_BRAND_NAME = 'Freezing Point'
""")
df_sql.show()

# Create a DataFrame from local Python data (useful for small test datasets)
data = [("lemonade", 4.50), ("ice cream sandwich", 6.00), ("bottled water", 2.00)]
df_local = session.create_dataframe(data, schema=["item", "price"])
df_local.show()

# Aggregate: group by truck brand and compute count, average price, and total menu value
df_summary = (
    df
    .group_by(col("TRUCK_BRAND_NAME"))
    .agg(
        count(col("MENU_ITEM_NAME")).alias("item_count"),
        avg(col("SALE_PRICE_USD")).alias("avg_price"),
        sum_(col("SALE_PRICE_USD")).alias("total_menu_value")
    )
    .sort(col("avg_price").desc())
)
df_summary.show()

# .describe() computes summary stats (count, mean, stddev, min, max) for selected columns
df.select(col("SALE_PRICE_USD"), col("COST_OF_GOODS_USD")).describe().show()

# Convert Snowpark DataFrame to a Pandas DataFrame (pulls data to local memory)
df_pandas = df_summary.to_pandas()
print(type(df_pandas))
df_pandas.head()

# Create the analytics schema if it doesn't already exist
session.sql("CREATE SCHEMA IF NOT EXISTS TASTY_BYTES_SAMPLE_DATA.analytics").collect()

# Write the summary DataFrame to a Snowflake table (overwrite if it already exists)
df_summary.write.mode("overwrite").save_as_table(
    "TASTY_BYTES_SAMPLE_DATA.analytics.brand_price_summary"
)

# Verify the table was created by reading it back
session.table("TASTY_BYTES_SAMPLE_DATA.analytics.brand_price_summary").show()



In [ ]:
import os
# Connection parameters for establishing a Snowpark session
# These environment variables must be set with real credentials before running
connection_params = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "password": os.environ["SNOWFLAKE_PASSWORD"],
    "role": "SYSADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "FROSTBYTE_TASTY_BYTES",
    "schema": "RAW_POS"
}

# Create a Snowpark session using the connection parameters
session = Session.builder.configs(connection_params).create()

# Create a DataFrame reference to the menu table (lazy evaluation – no data fetched yet)
df = session.table("TASTY_BYTES_SAMPLE_DATA.raw_pos.menu")

# Apply a filter and display results; .show() triggers execution and prints output
df.filter(col("TRUCK_BRAND_NAME") == "Freezing Point").show()

#### Shell Install Command For Snowpark Python Library
pip install snowflake-snowpark-python

**Tip:** This install is only needed when running Snowpark outside of Snowflake (e.g., in a local IDE). In Snowflake Notebooks, Snowpark is pre-installed and ready to use.

## Section: Snowpark DataFrames and VS Code Extension
### Lesson: Snowpark DataFrames - Code
Note: This code can only be run locally after performing the install with the above command in your local terminal or shell

**Tips:**
- When connecting from a local environment, you need a `Session` object configured with your account, user, password, role, warehouse, database, and schema.
- Keep credentials in a separate file (e.g., `credential.py`) and add it to `.gitignore` to avoid committing secrets.
- The VS Code Snowflake extension provides IntelliSense, syntax highlighting, and the ability to run queries directly from the editor.

In [ ]:
# Import the Snowpark Session class for connecting to Snowflake
from snowflake.snowpark import Session
import os

# Connection parameters – same pattern as above
# In production, use secrets management rather than environment variables
connection_params = {
    "account": os.environ["SNOWFLAKE_ACCOUNT"],
    "user": os.environ["SNOWFLAKE_USER"],
    "password": os.environ["SNOWFLAKE_PASSWORD"],
    "role": "SYSADMIN",
    "warehouse": "COMPUTE_WH",
    "database": "FROSTBYTE_TASTY_BYTES",
    "schema": "RAW_POS"
}

# Establish the Snowpark session
session = Session.builder.configs(connection_params).create()

# Reference the menu table as a Snowpark DataFrame
df = session.table("TASTY_BYTES_SAMPLE_DATA.raw_pos.menu")

# Filter for a specific truck brand and display results
df.filter(col("TRUCK_BRAND_NAME") == "Freezing Point").show()


## Section: Snowpark CLI
### Lesson: Snowpark CLI - Code

**Tips:**
- The **Snowflake CLI** (`snow`) lets you interact with Snowflake from your terminal — manage connections, deploy functions, and run queries.
- Install with `pip install snowflake-cli`.
- Use `snow connection add` to store connection profiles and `snow connection test` to verify them.
- The CLI is useful for CI/CD pipelines, automation scripts, and developers who prefer terminal workflows.

### This code should be run in your local terminal or shell, not a Snowflake Notebook
#### - Install the Snowflake CLI
pip install snowflake-cli

#### - Confirm the install worked
snow --version

#### - Add a new connection
snow connection add

#### - Test the connection
snow connection test --connection dev

#### - List all saved connections
snow connection list

**Tips:**
- `snow connection add` walks you through an interactive setup — have your account identifier, username, and role ready.
- Connection names (like `dev`) let you manage multiple environments (dev, staging, prod) easily.
- If `snow --version` fails, ensure your Python environment's bin/Scripts directory is in your PATH.

# Module 3

**Overview:** This module covers advanced topics including continuous data ingestion with Snowpipe, Generative AI with Snowflake Cortex, machine learning with Snowpark ML, and building applications with Streamlit in Snowflake.

## Section: Data Engineering with Snowflake
### Lesson: Snowpipe - Code
Note: This code will not run without valid AWS credentials, locations, and stages

**Tips:**
- **Snowpipe** enables continuous, event-driven data loading — files are ingested automatically as they arrive in a stage.
- Snowpipe uses a `COPY INTO` statement defined inside a `CREATE PIPE` object.
- `AUTO_INGEST=TRUE` means Snowpipe listens for cloud event notifications (e.g., S3 SQS events) to trigger loading.
- A **storage integration** securely connects Snowflake to your cloud storage without embedding credentials in SQL.
- After creating a storage integration, run `DESCRIBE INTEGRATION` to get the IAM values needed for your cloud provider trust policy.
- Use `SHOW PIPES` and `DESCRIBE PIPE` to inspect pipe configuration.
- Pause pipes with `ALTER PIPE ... SET PIPE_EXECUTION_PAUSED = TRUE` before dropping them to avoid orphaned notifications.

In [ ]:
%%sql -r dataframe_19
---> create the storage integration
-- A storage integration securely connects Snowflake to an external cloud storage (S3 here)
-- It uses an IAM role ARN to authenticate without storing credentials in Snowflake
CREATE OR REPLACE STORAGE INTEGRATION S3_role_integration
  TYPE = EXTERNAL_STAGE
  STORAGE_PROVIDER = S3
  ENABLED = TRUE
  STORAGE_AWS_ROLE_ARN = "REMOVED"
  STORAGE_ALLOWED_LOCATIONS = ("s3://intro-to-snowflake-snowpipe/");

---> describe the storage integration to see the info you need to copy over to AWS
-- Copy STORAGE_AWS_IAM_USER_ARN and STORAGE_AWS_EXTERNAL_ID to your AWS trust policy
DESCRIBE INTEGRATION S3_role_integration;

---> create the database
CREATE OR REPLACE DATABASE S3_db;

---> create the table (automatically in the public schema, because we didn't specify)
CREATE OR REPLACE TABLE S3_table(food STRING, taste INT);

USE SCHEMA S3_db.public;

---> create stage with the link to the S3 bucket and info on the associated storage integration
-- The stage combines the S3 URL with the integration for secure access
CREATE OR REPLACE STAGE S3_stage
  url = ('s3://intro-to-snowflake-snowpipe/')
  storage_integration = S3_role_integration;

SHOW STAGES;

---> see the files in the stage
LIST @S3_stage;

---> select the first two columns from the stage
-- You can query staged files directly using positional column references ($1, $2, etc.)
SELECT $1, $2 FROM @S3_stage;

USE WAREHOUSE COMPUTE_WH;

---> create the snowpipe, copying from S3_stage into S3_table
-- Snowpipe enables continuous, event-driven data loading (AUTO_INGEST=TRUE uses S3 event notifications)
CREATE PIPE S3_db.public.S3_pipe AUTO_INGEST=TRUE as
  COPY INTO S3_db.public.S3_table
  FROM @S3_db.public.S3_stage;

-- Check if any data has been loaded
SELECT * FROM S3_db.public.S3_table;

---> see a list of all the pipes
SHOW PIPES;

-- DESCRIBE PIPE shows the pipe definition and notification channel ARN
DESCRIBE PIPE S3_db.public.S3_pipe;

---> pause the pipe
-- Pausing stops Snowpipe from loading new files (existing in-flight loads complete)
ALTER PIPE S3_db.public.S3_pipe SET PIPE_EXECUTION_PAUSED = TRUE;

---> drop the pipe
DROP PIPE S3_pipe;

SHOW PIPES;

## Section: GenAI with Snowflake
### Lesson: GenAI Overview Part 2
Note: This code will not work in a trial account. You must use a paid, non-trial account to use the Cortex Search Service

**Tips:**
- **Cortex Search** provides a managed RAG (Retrieval-Augmented Generation) service — it indexes your text data and lets you query it with natural language.
- It automatically handles chunking, embedding, and vector search behind the scenes.
- Use it when you need semantic search over documents, support tickets, knowledge bases, etc.
- Cortex Search works with structured text columns in Snowflake tables — no need to manage external vector databases.

In [ ]:
%%sql -r dataframe_16
-- Create a Cortex Search Service for full-text semantic search over menu items
-- ON menu_item_name: the column to index for search
-- TARGET_LAG: how fresh the index should be (refreshes within 1 hour of source changes)
-- The AS query defines which columns are indexed and returned in search results
CREATE OR REPLACE CORTEX SEARCH SERVICE tasty_bytes.analytics.tasty_bytes_menu_search
  ON menu_item_name
  WAREHOUSE = compute_wh
  TARGET_LAG = '1 hour'
AS
SELECT menu_item_name, truck_brand_name, item_category
FROM tasty_bytes.raw_pos.menu;


## Section: GenAI with Snowflake
### Lesson: Cortex CLI

**Tips:**
- The `snow cortex` CLI commands let you use LLM functions directly from your terminal.
- `snow cortex complete` sends a prompt to a model and returns the completion.
- `snow cortex translate` translates text between languages.
- You can pipe text from files with `cat file.txt | snow cortex complete "prompt"` for batch processing.
- Chain commands together to build multi-step AI pipelines (e.g., summarize then translate).
- These commands run against your Snowflake account — the data stays within Snowflake's security boundary.

## Section: GenAI with Snowflake
### Lesson: Cortex CLI

1) snow --version

2) snow cortex --help

3) snow cortex complete "What are three reasons Snowflake is well suited for AI workloads?"

4) snow cortex complete "What are three reasons Snowflake is well suited for AI workloads?" 
  --model snowflake-arctic

5) cat sample_review.txt

6) cat sample_review.txt | snow cortex complete \
  "Summarize this review in one sentence and classify the sentiment:" 
  --model snowflake-arctic

7) result=$(snow cortex complete \
  "Give me a one-word sentiment: positive, negative, or neutral. Review: Great product, fast delivery." \
  --model snowflake-arctic)
echo "Sentiment was: $result"

## Section: GenAI with Snowflake
### Lesson: Snowflake Cortex LLM Functions - Code
Note: This code will not work in a trial account. You must use a paid, non-trial account to use the Cortex Complete function

**Tips:**
- **Cortex LLM Functions** are SQL-native AI functions you call directly in queries: `SNOWFLAKE.CORTEX.COMPLETE()`, `SNOWFLAKE.CORTEX.SENTIMENT()`, `SNOWFLAKE.CORTEX.SUMMARIZE()`, `SNOWFLAKE.CORTEX.TRANSLATE()`, etc.
- They process data row-by-row, so you can apply AI to every row in a table with a simple SELECT.
- No model deployment or infrastructure management needed — just call the function.
- `COMPLETE()` takes a model name and a prompt, and returns the generated text.
- These functions keep your data inside Snowflake's governance perimeter — data never leaves your account.

In [ ]:
%%sql -r dataframe_20
---> use the llama3.1-8b model and Snowflake Cortex Complete to ask a question
-- CORTEX.COMPLETE() sends a prompt to an LLM and returns the generated text
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
    'llama3.1-8b', 'What are three reasons that Snowflake is positioned to become the go-to data platform?');

---> now send the result to the Snowflake Cortex Summarize function
-- CORTEX.SUMMARIZE() condenses long text into a brief summary
-- Here we chain Complete's output directly into Summarize
SELECT SNOWFLAKE.CORTEX.AI_SUMMARIZE(SNOWFLAKE.CORTEX.COMPLETE(
    'llama3.1-8b', 'What are three reasons that Snowflake is positioned to become the go-to data platform?'));

---> run Snowflake Cortex Complete on multiple rows at once
-- CONCAT builds a prompt for each row; Complete processes each one independently
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
    'llama3.1-8b',
        CONCAT('Tell me why this food is tasty: ', menu_item_name)
) FROM TASTY_BYTES_SAMPLE_DATA.RAW_POS.MENU LIMIT 5;

---> check out what the table of prompts we're feeding to Complete (roughly) looks like
SELECT CONCAT('Tell me why this food is tasty: ', menu_item_name)
FROM TASTY_BYTES_SAMPLE_DATA.RAW_POS.MENU LIMIT 5;

---> give Snowflake Cortex Complete a prompt with history
-- Multi-turn conversation format: system sets context, user provides input
-- The array of role/content objects simulates a chat history
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
    'llama3.1-8b',
    [
        {'role': 'system', 
        'content': 'Analyze this Snowflake review and determine the overall sentiment. Answer with just \"Positive\", \"Negative\", or \"Neutral\"' },
        {'role': 'user',
        'content': 'I love Snowflake because it is so simple to use.'}
    ],
    {}
) AS response;

---> give Snowflake Cortex Complete a prompt with a lengthier history
-- Adding assistant's previous response enables multi-turn follow-up questions
SELECT SNOWFLAKE.CORTEX.AI_COMPLETE(
    'llama3.1-8b',
    [
        {'role': 'system', 
        'content': 'Analyze this Snowflake review and determine the overall sentiment. Answer with just \"Positive\", \"Negative\", or \"Neutral\"' },
        {'role': 'user',
        'content': 'I love Snowflake because it is so simple to use.'},
        {'role': 'assistant',
        'content': 'Positive. The review expresses a positive sentiment towards Snowflake, specifically mentioning that it is \"so simple to use.\'"'},
        {'role': 'user',
        'content': 'Based on other information you know about Snowflake, explain why the reviewer might feel they way they do.'}
    ],
    {}
) AS response;

## Section: Machine Learning with Snowflake
### Lesson: Snowpark ML Modeling - Code
Note: The following code is meant to be run in an external Jupyter notebook that has a local credential.py file to hold login credentials. In a Snowflake Notebook, the session is already available and no external credentials are needed.

**Tips:**
- **Snowpark ML** provides scikit-learn-compatible APIs (preprocessing, modeling) that run computations inside Snowflake.
- Use `LabelEncoder`, `OneHotEncoder`, `StandardScaler`, etc. from `snowflake.ml.modeling.preprocessing`.
- Models like `XGBClassifier` from `snowflake.ml.modeling.xgboost` train on Snowflake compute — no data extraction needed.
- `model.fit(train_df)` trains the model; `model.score(test_df)` evaluates accuracy.
- Use `randomSplit([0.9, 0.1])` to split a Snowpark DataFrame into train/test sets.
- In Snowflake Notebooks, replace `Session.builder.configs(params).create()` with `get_active_session()`.

In [ ]:
# Note: This is not code you can run in a SQL worksheet. We ran this in a Jupyter notebook

# Install the Snowflake ML and Snowpark packages needed for model training
!pip install snowflake-ml-python
!pip install snowflake-snowpark-python

# Import credentials from a separate file (keeps secrets out of source code)
from credential import params

# if you want guidance on connecting to Snowflake from your IDE, see here:
# https://docs.snowflake.com/en/developer-guide/snowpark/python/creating-session#creating-a-session

# Import required libraries for Snowpark, ML, and data manipulation
from snowflake.snowpark import Session
from snowflake.ml.modeling.xgboost import XGBClassifier
from snowflake.snowpark.functions import col
from snowflake.ml.modeling import preprocessing

# Here's the neighborhood visiting pattern the truck follows:
# In January, the truck goes to N1 on the 1st, 8th, 15th, 22nd, and 29th, and N2 the other days.

# From February through November, it goes to:
# N1 on the 1st
# N2 on the 2nd
# N3 on the 3rd
# N4 on the 4th
# N5 on the 5th
# N6 on the 6th
# N7 on the 7th
# N1 on the 8th
# N2 on the 9th
# etc.

# Every December, it only goes to N8.

# Number of days in each month (non-leap year)
month_days = [31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31]

# Build a dictionary mapping (month, day) tuples to neighborhood numbers
pre = {}

for i,month_length in enumerate(month_days):
    month = i + 1

    for day in range(1,month_length+1):
        
        # January: neighborhood 1 every 7th day starting from day 1, otherwise neighborhood 2
        if ((month) == 1):
            if (day) % 7 == 1:
                pre[(month,day)] = 1
            else:
                pre[(month,day)] = 2
                
        # Feb-Nov: cycle through neighborhoods 1-7 based on day of month
        elif ((month) <= 11):
            pre[(month,day)] = ((day-1) % 7) + 1

        # December: always neighborhood 8
        elif ((month) == 12):
            pre[(month,day)] = 8

# Preview the mapping dictionary
pre

# Note: Here, I skipped the step of uploading the final "df_clean" dataset to Snowflake

# Create a Snowpark session to interact with Snowflake
session = Session.builder.configs(params).create()

# Create a lazy DataFrame reference to the cleaned dataset in Snowflake
# (no data is pulled into local memory until an action like .show() is called)
snowpark_df = session.table("test_database.test_schema.df_clean")

# Display first 40 rows
snowpark_df.show(n=40)

# Count total rows in the dataset
snowpark_df.count()

# Show summary statistics (count, mean, stddev, min, max) for numeric columns
snowpark_df.describe().show()

# See how many records exist per neighborhood (class distribution)
snowpark_df.group_by("Neighborhood").count().show()

# Manual approach: subtract 1 from neighborhood to make it 0-indexed for XGBoost
test = snowpark_df.withColumn('NEIGHBORHOOD2', snowpark_df.neighborhood - 1).drop("Neighborhood")

test.show()

# Better approach: use Snowpark ML's LabelEncoder to encode the target variable
# LabelEncoder maps categorical labels to integers (0, 1, 2, ...)
le = LabelEncoder(input_cols=['NEIGHBORHOOD'], output_cols= ['NEIGHBORHOOD2'], drop_input_cols=True)

# Fit the encoder on the neighborhood column to learn the mapping
fitted = le.fit(snowpark_df.select("NEIGHBORHOOD"))

# Transform the DataFrame to replace NEIGHBORHOOD with encoded NEIGHBORHOOD2
snowpark_df_prepared = fitted.transform(snowpark_df)

snowpark_df_prepared.show()

# Split data: 90% for training, 10% for testing
train_snowpark_df, test_snowpark_df = snowpark_df_prepared.randomSplit([0.9, 0.1])

# Persist training data to a Snowflake table for reproducibility
train_snowpark_df.write.mode("overwrite").save_as_table("df_clean_train")

# Persist test data to a Snowflake table
test_snowpark_df.write.mode("overwrite").save_as_table("df_clean_test")

# Define feature columns (inputs) and label column (target to predict)
FEATURE_COLS = ["MONTH", "DAY"]
LABEL_COLS = ["NEIGHBORHOOD2"]

# Create and train an XGBoost classifier using Snowpark ML
# Training happens inside Snowflake (pushdown computation)
xgboost_model = XGBClassifier(
    input_cols=FEATURE_COLS,
    label_cols=LABEL_COLS
)

# Fit the model on the training data
xgboost_model.fit(train_snowpark_df)

# Evaluate model accuracy on the held-out test set
accuracy = xgboost_model.score(test_snowpark_df)

print("Accuracy: %.2f%%" % (accuracy * 100.0))

## Section: Applications with Snowflake
### Lesson: Streamlit in Snowflake - Code
Note: Streamlit is not supported inside Snowflake Notebooks. The following code is meant to be used inside a Snowflake app.

**Tips:**
- **Streamlit in Snowflake (SiS)** lets you build interactive data apps directly in Snowsight — no separate hosting needed.
- Create a Streamlit app from the Snowsight UI: Projects > Streamlit > + Streamlit App.
- Your app runs inside Snowflake, so it has direct access to your data with full governance and security.
- Common components: `st.dataframe()` for tables, `st.bar_chart()` for charts, `st.selectbox()` for filters, `st.text_input()` for user input.
- Use `get_active_session()` inside your Streamlit app to query Snowflake data.
- Apps can be shared with other Snowflake users via role-based access.

In [ ]:
# Import required packages for the Streamlit app
import pandas as pd
import streamlit as st
from snowflake.snowpark.context import get_active_session
import altair as alt

# Get the active Snowpark session (provided by Streamlit in Snowflake)
session = get_active_session()

# --- Streamlit App Layout ---
st.title(":snowflake: Tasty Bytes Streamlit App :snowflake:")
st.write(
    """Tasty Bytes is a fictitious, global food truck network, that is on a mission to serve unique food options with high quality items in a safe, convenient and cost effective way. In order to drive
forward on their mission, Tasty Bytes is beginning to leverage the Snowflake Data Cloud.
    """
)
st.divider()


# Cache data to avoid re-running the query on every Streamlit rerun
@st.cache_data
def get_city_sales_data(city_names: list, start_year: int = 2020, end_year: int = 2023):
    """Query total daily sales for selected cities within a year range."""
    sql = f"""
        SELECT
            date,
            primary_city,
            SUM(order_total) AS sum_orders
        FROM tasty_bytes.analytics.orders_v
        WHERE primary_city in ({city_names})
            and year(date) between {start_year} and {end_year}
        GROUP BY date, primary_city
        ORDER BY date DESC
    """
    sales_data = session.sql(sql).to_pandas()
    return sales_data, sql

@st.cache_data
def get_unique_cities():
    """Fetch distinct city names for the multi-select dropdown."""
    sql = """
        SELECT DISTINCT primary_city
        FROM tasty_bytes.analytics.orders_v
        ORDER BY primary_city
    """
    city_data = session.sql(sql).to_pandas()
    return city_data

def get_city_sales_chart(sales_data: pd.DataFrame):
    """Build an Altair line chart showing daily sales by city."""
    sales_data["SUM_ORDERS"] = pd.to_numeric(sales_data["SUM_ORDERS"])
    sales_data["DATE"] = pd.to_datetime(sales_data["DATE"])

    # Create a multi-line chart colored by city
    chart = (
        alt.Chart(sales_data)
        .mark_line(point=False, tooltip=True)
        .encode(
            alt.X("DATE", title="Date"),
            alt.Y("SUM_ORDERS", title="Total Orders Sum USD"),
            color="PRIMARY_CITY",
        )
    )
    return chart


def format_sql(sql):
    """Remove leading whitespace from SQL for cleaner display."""
    return sql.replace("\n        ", "\n")


# --- Interactive Filters ---
first_col, second_col = st.columns(2, gap="large")

with first_col:
    # Slider to select the year range for filtering sales data
    start_year, end_year = st.select_slider(
        "Select date range you want to filter the chart on below:",
        options=range(2020, 2024),
        value=(2020, 2023),
    )
with second_col:
    # Multi-select widget for choosing which cities to display
    selected_city = st.multiselect(
        label="Select cities below that you want added to the chart below:",
        options=get_unique_cities()["PRIMARY_CITY"].tolist(),
        default="San Mateo",
    )

# Build a comma-separated, quoted list of city names for the SQL IN clause
if len(selected_city) == 0:
    city_selection = ""
else:
    city_selection = selected_city
city_selection_list = ("'" + "','".join(city_selection) + "'") if city_selection else ""

# Fetch data and build the chart
sales_data, sales_sql = get_city_sales_data(city_selection_list, start_year, end_year)
sales_fig = get_city_sales_chart(sales_data)

# --- Display Results in Tabs ---
chart_tab, dataframe_tab, query_tab = st.tabs(["Chart", "Raw Data", "SQL Query"])
chart_tab.altair_chart(sales_fig, use_container_width=True)
dataframe_tab.dataframe(sales_data, use_container_width=True)
query_tab.code(format_sql(sales_sql), "sql")
